# Flickr30k S1 baseline — seed 2026

Huấn luyện baseline **S1: MobileNetV3-Large** trên Flickr30k bằng
seed **2026** (confirmation).

## Kaggle inputs bắt buộc — đúng 5 input

1. Output `F30K_Data`;
2. output `F30K_Base`;
3. output `02A_Phase2_Shared_Pipeline`;
4. dataset ảnh `Flickr Image dataset`;
5. package/dataset `pycocoevalcap-1.2` offline.

Khi resume, gắn thêm **một** output gần nhất của chính notebook
`F30K_S1_2026`.

Không gắn MS COCO, Teacher, output KD, Test caption hoặc baseline model khác.

## Protocol khóa

- tối đa 20 epoch;
- Early Stopping: bắt đầu từ epoch 5, patience 4, min delta CIDEr-D = 0,001;
- chọn duy nhất bằng Validation CIDEr-D;
- Beam-3, length penalty 0,7;
- physical/effective batch: 32/64;
- Test: `SEALED / NOT USED`;
- GPU T4, Internet Off.


## Cell 1 — Authorization, integrity và Test-seal

In [1]:

import csv, hashlib, importlib.util, json, math, os, random, shutil, subprocess, sys, time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

MODEL_ID = "S1"
MODEL_NAME = "MobileNetV3-Large"
SEED = 2026
EXPERIMENT_ID = f"F30K_{MODEL_ID}_{SEED}"
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def unique_named(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (f"Cần đúng một {name}", matches)
    return matches[0]

BASE_READY_PATH = unique_named("F30K_BASE_READY.json")
DATA_CONTRACT_PATH = unique_named("F30K_DATA_CONTRACT.json")
PROTOCOL_PATH = unique_named("F30K_BASELINE_PROTOCOL.json")
TRAIN_PATH = unique_named("f30k_train.parquet")
VAL_PATH = unique_named("f30k_validation.parquet")
VOCAB_PATH = unique_named("f30k_vocabulary_minfreq5.json")
ENCODER_INIT_PATH = unique_named("s1_encoder_adapter_initial_seed42.pt")

base_ready = json.loads(BASE_READY_PATH.read_text(encoding="utf-8"))
data_contract = json.loads(DATA_CONTRACT_PATH.read_text(encoding="utf-8"))
protocol = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
vocab = json.loads(VOCAB_PATH.read_text(encoding="utf-8"))

assert base_ready["status"] == "authorized"
assert base_ready["test_set_status"] == "sealed"
assert base_ready["test_set_used"] is False
assert (
    SEED == int(protocol["development_seed"])
    or SEED in [int(x) for x in protocol["confirmation_seeds"]]
)
assert protocol["vocabulary_size"] == 7456 == len(vocab["tokens"])
assert protocol["maximum_epochs"] == 20
assert protocol["selection_metric"] == "Validation CIDEr-D"
assert data_contract["test_set_status"] == "sealed"
assert data_contract["test_caption_exported"] is False

for file_name, expected in {
    "f30k_train.parquet": data_contract["artifacts"]["train"]["sha256"],
    "f30k_validation.parquet": data_contract["artifacts"]["validation"]["sha256"],
    "f30k_vocabulary_minfreq5.json": data_contract["artifacts"]["vocabulary"]["sha256"],
}.items():
    actual_path = {"f30k_train.parquet": TRAIN_PATH,
                   "f30k_validation.parquet": VAL_PATH,
                   "f30k_vocabulary_minfreq5.json": VOCAB_PATH}[file_name]
    assert sha256(actual_path) == expected, file_name

if importlib.util.find_spec("pycocoevalcap") is None:
    wheels = sorted(INPUT_ROOT.rglob("pycocoevalcap*.whl"))
    if wheels:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", str(wheels[0])])
    else:
        packages = [p for p in INPUT_ROOT.rglob("pycocoevalcap") if p.is_dir()]
        assert packages, "Hãy gắn input pycocoevalcap-1.2 offline."
        sys.path.insert(0, str(packages[0].parent))

assert importlib.util.find_spec("pycocoevalcap") is not None
assert shutil.which("java"), "METEOR chính thức yêu cầu Java runtime."
assert torch.cuda.is_available(), "Notebook này yêu cầu GPU T4."
DEVICE = torch.device("cuda")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

print("Preflight             : PASSED")
print("Experiment            :", EXPERIMENT_ID)
print("Model                 :", MODEL_NAME)
print("Train/Validation      : VERIFIED")
print("Vocabulary            :", len(vocab["tokens"]))
print("Selection             : Validation CIDEr-D")
print("Test                   : SEALED / NOT READ")


Preflight             : PASSED
Experiment            : F30K_S1_2026
Model                 : MobileNetV3-Large
Train/Validation      : VERIFIED
Vocabulary            : 7456
Selection             : Validation CIDEr-D
Test                   : SEALED / NOT READ


## Cell 2 — Flickr30k loaders, encoder và decoder mới

In [2]:

train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)
assert train_df["image_id"].nunique() == 29000 and len(train_df) == 145000
assert val_df["image_id"].nunique() == 1014 and len(val_df) == 5070
assert set(train_df.image_id).isdisjoint(set(val_df.image_id))

needed_names = {f"{int(x)}.jpg" for x in pd.concat([train_df.image_id, val_df.image_id])}
image_candidates = [p for p in INPUT_ROOT.rglob("*.jpg") if p.name in needed_names]
IMAGE_BY_NAME = {}
duplicate_alias_count = 0
for path in image_candidates:
    old = IMAGE_BY_NAME.get(path.name)
    if old is None:
        IMAGE_BY_NAME[path.name] = path
        continue
    if old.resolve() == path.resolve():
        continue
    # Một số bản đóng gói Kaggle lưu cùng ảnh ở cả
    # flickr30k_images/<file> và flickr30k_images/flickr30k_images/<file>.
    # Chấp nhận alias lồng nhau khi kích thước byte giống hệt và luôn chọn
    # đường dẫn nông/ngắn nhất. Nếu kích thước khác nhau, dừng để tránh ánh xạ
    # nhầm hai ảnh chỉ tình cờ có cùng tên.
    assert old.stat().st_size == path.stat().st_size, (
        "Hai ảnh cùng tên nhưng khác kích thước", path.name, old, path
    )
    duplicate_alias_count += 1
    IMAGE_BY_NAME[path.name] = min(
        [old, path],
        key=lambda p: (len(p.parts), len(str(p)), str(p)),
    )
missing = sorted(needed_names - set(IMAGE_BY_NAME))
assert not missing, ("Thiếu ảnh Flickr30k", len(missing), missing[:10])

train_groups = []
for image_id, group in train_df.groupby("image_id", sort=True):
    train_groups.append((int(image_id), [list(map(int, x)) for x in group.token_ids]))
val_groups = []
VAL_REFERENCES = {}
for image_id, group in val_df.groupby("image_id", sort=True):
    iid = int(image_id)
    val_groups.append((iid, [list(map(int, x)) for x in group.token_ids]))
    VAL_REFERENCES[iid] = [str(x) for x in group.caption.tolist()]

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0), interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class CaptionDataset(Dataset):
    def __init__(self, rows, transform, training):
        self.rows, self.transform, self.training, self.epoch = rows, transform, training, 0
    def set_epoch(self, epoch):
        self.epoch = int(epoch)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        image_id, captions = self.rows[index]
        caption = captions[(self.epoch + image_id % len(captions)) % len(captions)] if self.training else captions[0]
        with Image.open(IMAGE_BY_NAME[f"{image_id}.jpg"]) as im:
            image = self.transform(im.convert("RGB"))
        return image_id, image, torch.tensor(caption, dtype=torch.long)

def collate(batch):
    image_ids, images, captions = zip(*batch)
    max_len = max(len(x) for x in captions)
    tokens = torch.zeros(len(captions), max_len, dtype=torch.long)
    mask = torch.zeros(len(captions), max_len, dtype=torch.bool)
    for i, x in enumerate(captions):
        tokens[i, :len(x)] = x
        mask[i, :len(x)] = True
    return torch.tensor(image_ids), torch.stack(images), tokens, mask

train_set = CaptionDataset(train_groups, train_transform, True)
val_set = CaptionDataset(val_groups, val_transform, False)

def make_train_loader(epoch):
    train_set.set_epoch(epoch)
    g = torch.Generator().manual_seed(SEED + epoch)
    return DataLoader(train_set, batch_size=32, shuffle=True, generator=g, num_workers=2,
                      pin_memory=True, persistent_workers=False, collate_fn=collate)

val_loader = DataLoader(val_set, batch_size=64, shuffle=False, num_workers=2,
                        pin_memory=True, persistent_workers=True, collate_fn=collate)

def select_tensor(x):
    if torch.is_tensor(x):
        return x
    if isinstance(x, dict):
        return [v for v in x.values() if torch.is_tensor(v)][-1]
    if isinstance(x, (tuple, list)):
        return [v for v in x if torch.is_tensor(v)][-1]
    raise TypeError(type(x))

class VisualEncoder(nn.Module):
    def __init__(self, backbone, source_dim):
        super().__init__()
        self.backbone = backbone
        self.source_dim = source_dim
        self.projection = nn.Linear(source_dim, 256)
        self.visual_position_embedding = nn.Parameter(torch.empty(1, 49, 256))
        self.output_norm = nn.LayerNorm(256)
        nn.init.xavier_uniform_(self.projection.weight)
        nn.init.zeros_(self.projection.bias)
        nn.init.normal_(self.visual_position_embedding, std=0.02)
    def forward(self, images):
        if MODEL_ID == "S1":
            feat = self.backbone(images)
        else:
            feat = select_tensor(self.backbone.forward_features(images))
        assert feat.ndim == 4 and feat.shape[1] == self.source_dim and tuple(feat.shape[-2:]) == (7, 7), feat.shape
        tokens = feat.flatten(2).transpose(1, 2).contiguous()
        return self.output_norm(self.projection(tokens) + self.visual_position_embedding)

class CaptionDecoder(nn.Module):
    def __init__(self, vocabulary_size):
        super().__init__()
        self.vocabulary_size = vocabulary_size
        self.d_model, self.max_steps = 256, 31
        self.token_embedding = nn.Embedding(vocabulary_size, 256, padding_idx=0)
        self.position_embedding = nn.Embedding(31, 256)
        self.embedding_dropout = nn.Dropout(0.1)
        layer = nn.TransformerDecoderLayer(256, 8, 1024, 0.1, "gelu",
                                           batch_first=True, norm_first=True)
        self.transformer_decoder = nn.TransformerDecoder(layer, 3)
        self.final_norm = nn.LayerNorm(256)
        self.output_bias = nn.Parameter(torch.zeros(vocabulary_size))
        nn.init.normal_(self.token_embedding.weight, std=256 ** -0.5)
        nn.init.normal_(self.position_embedding.weight, std=0.02)
        with torch.no_grad():
            self.token_embedding.weight[0].zero_()
    def forward(self, token_ids, memory, attention_mask=None):
        n, t = token_ids.shape
        pos = torch.arange(t, device=token_ids.device).unsqueeze(0).expand(n, -1)
        x = self.embedding_dropout(self.token_embedding(token_ids) * 16.0 + self.position_embedding(pos))
        causal = torch.triu(torch.ones(t, t, dtype=torch.bool, device=token_ids.device), diagonal=1)
        key_pad = None if attention_mask is None else ~attention_mask.bool()
        x = self.transformer_decoder(x, memory, tgt_mask=causal, tgt_key_padding_mask=key_pad)
        return F.linear(self.final_norm(x), self.token_embedding.weight, self.output_bias)

if MODEL_ID == "S1":
    from torchvision.models import mobilenet_v3_large
    base = mobilenet_v3_large(weights=None).features
elif MODEL_ID == "S2":
    import timm
    base = timm.create_model("mobilevit_s.cvnets_in1k", pretrained=False)
    base.reset_classifier(num_classes=0, global_pool="")
    for head_name in ["head", "head_dist", "classifier", "fc"]:
        head = getattr(base, head_name, None)
        if isinstance(head, nn.Module) and not isinstance(head, nn.Identity):
            setattr(base, head_name, nn.Identity())
elif MODEL_ID == "S3":
    import timm
    base = timm.create_model("tiny_vit_5m_224.dist_in22k_ft_in1k", pretrained=False)
    base.reset_classifier(num_classes=0, global_pool="")
    for head_name in ["head", "head_dist", "classifier", "fc"]:
        head = getattr(base, head_name, None)
        if isinstance(head, nn.Module) and not isinstance(head, nn.Identity):
            setattr(base, head_name, nn.Identity())
else:
    raise ValueError(MODEL_ID)

encoder = VisualEncoder(base, 960)
initial = torch.load(ENCODER_INIT_PATH, map_location="cpu", weights_only=False)
encoder.load_state_dict(initial["state_dict"], strict=True)

# Decoder Flickr30k được khởi tạo mới bằng SEED hiện tại; không nạp decoder
# MS COCO có vocabulary 10.080.
torch.manual_seed(SEED)
decoder = CaptionDecoder(len(vocab["tokens"]))
encoder.to(DEVICE)
decoder.to(DEVICE)

for module in encoder.modules():
    if isinstance(module, nn.modules.batchnorm._BatchNorm):
        module.eval()

enc_params = sum(p.numel() for p in encoder.parameters())
dec_params = sum(p.numel() for p in decoder.parameters())
assert enc_params == 3231024, (enc_params, 3231024)
assert dec_params == 5_084_960, dec_params
print("Images train/val       :", len(train_groups), "/", len(val_groups))
print("Nested image aliases   :", duplicate_alias_count, "(deduplicated)")
print("Encoder/decoder params :", enc_params, "/", dec_params)
print("Test                   : SEALED / NOT READ")


Images train/val       : 29000 / 1014
Nested image aliases   : 30014 (deduplicated)
Encoder/decoder params : 3231024 / 5084960
Test                   : SEALED / NOT READ


## Cell 3 — Beam-3 và metrics Validation

In [3]:

TOKENS = vocab["tokens"]
SPECIAL = {0, 1, 2, 3}

def ids_to_caption(ids):
    words = []
    for x in ids:
        x = int(x)
        if x == 2:
            break
        if x not in SPECIAL:
            words.append(TOKENS[x])
    text = " ".join(words)
    for p in [".", ",", "!", "?", ";", ":"]:
        text = text.replace(" " + p, p)
    return text.strip()

@torch.inference_mode()
def beam_decode(memory, beam_size=3, alpha=0.7):
    beams = [([1], 0.0, False)]
    for _ in range(30):
        active = [b for b in beams if not b[2]]
        if not active:
            break
        seq = torch.tensor([b[0] for b in active], device=DEVICE)
        mem = memory.expand(len(active), -1, -1)
        logits = decoder(seq, mem)[:, -1].float()
        logits[:, 0] = -torch.inf
        logits[:, 1] = -torch.inf
        logp = logits.log_softmax(-1)
        candidates = [b for b in beams if b[2]]
        for i, (tokens, score, _) in enumerate(active):
            values, indices = logp[i].topk(beam_size)
            for value, index in zip(values.tolist(), indices.tolist()):
                new = tokens + [index]
                candidates.append((new, score + value, index == 2))
        beams = sorted(candidates, key=lambda b: b[1] / (max(1, len(b[0]) - 1) ** alpha), reverse=True)[:beam_size]
    return max(beams, key=lambda b: b[1] / (max(1, len(b[0]) - 1) ** alpha))[0]

@torch.inference_mode()
def validate():
    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.meteor.meteor import Meteor
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.cider.cider import Cider
    encoder.eval()
    decoder.eval()
    hypotheses = {}
    ce_sum, token_count = 0.0, 0
    for image_ids, images, tokens, mask in val_loader:
        images, tokens, mask = images.to(DEVICE), tokens.to(DEVICE), mask.to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16):
            memory = encoder(images)
            logits = decoder(tokens[:, :-1], memory, mask[:, :-1])
        targets = tokens[:, 1:]
        valid = mask[:, 1:]
        losses = F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]),
                                 targets.reshape(-1), reduction="none").reshape_as(targets)
        ce_sum += float((losses * valid).sum())
        token_count += int(valid.sum())
        for j, image_id in enumerate(image_ids.tolist()):
            hypotheses[int(image_id)] = [ids_to_caption(beam_decode(memory[j:j+1]))]
    refs = {int(k): [{"caption": x} for x in v] for k, v in VAL_REFERENCES.items()}
    hyps = {int(k): [{"caption": v[0]}] for k, v in hypotheses.items()}
    tokenizer = PTBTokenizer()
    refs = tokenizer.tokenize(refs)
    hyps = tokenizer.tokenize(hyps)
    b, _ = Bleu(4).compute_score(refs, hyps)
    meteor = Meteor()
    try:
        m, _ = meteor.compute_score(refs, hyps)
    finally:
        close_method = getattr(meteor, "close", None)
        if callable(close_method):
            close_method()
    r, _ = Rouge().compute_score(refs, hyps)
    c, _ = Cider().compute_score(refs, hyps)
    return {"ce_loss": ce_sum / token_count, "BLEU_1": b[0], "BLEU_4": b[3],
            "METEOR": m, "ROUGE_L": r, "CIDEr_D": c}, hypotheses


## Cell 4 — Full train, Early Stopping và exact resume

In [4]:

MAX_EPOCHS = int(protocol["maximum_epochs"])
PATIENCE = int(protocol["early_stopping"]["patience"])
MIN_EPOCHS = int(protocol["early_stopping"]["minimum_completed_epochs"])
MIN_DELTA = float(protocol["minimum_cider_improvement"])
ACCUM = 2
steps_per_epoch = math.ceil(math.ceil(len(train_set) / 32) / ACCUM)
total_steps = MAX_EPOCHS * steps_per_epoch
warmup = max(1, round(0.05 * total_steps))

optimizer = torch.optim.AdamW([
    {"params": encoder.backbone.parameters(), "lr": 1e-4},
    {"params": list(encoder.projection.parameters()) + [encoder.visual_position_embedding] + list(encoder.output_norm.parameters()), "lr": 3e-4},
    {"params": decoder.parameters(), "lr": 3e-4},
], betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-4)

def lr_lambda(step):
    if step < warmup:
        return max(1e-8, step / warmup)
    progress = min(1.0, (step - warmup) / max(1, total_steps - warmup))
    return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", init_scale=1024.0)
LAST = WORK_ROOT / "last.pt"
BEST = WORK_ROOT / "best.pt"
METRICS_CSV = WORK_ROOT / "metrics.csv"

def state_cpu(module):
    return {k: v.detach().cpu() for k, v in module.state_dict().items()}

def save_checkpoint(path, epoch, best, stale, global_step, metrics):
    payload = {
        "schema_version": "f30k_student_baseline_v1",
        "experiment_id": EXPERIMENT_ID, "model_id": MODEL_ID, "seed": SEED,
        "epoch": epoch, "best_cider": best, "stale_evaluations": stale,
        "global_step": global_step, "metrics": metrics,
        "encoder_state_dict": state_cpu(encoder),
        "decoder_state_dict": state_cpu(decoder),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": torch.cuda.get_rng_state_all(),
        "protocol_sha256": sha256(PROTOCOL_PATH),
        "data_contract_sha256": sha256(DATA_CONTRACT_PATH),
        "test_set_status": "sealed", "test_set_used": False,
    }
    tmp = path.with_suffix(".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)

start_epoch, best_cider, stale, global_step = 0, -math.inf, 0, 0
resume_candidates = sorted(INPUT_ROOT.rglob(f"{EXPERIMENT_ID}/last.pt"))
resume_metrics_path = None
if resume_candidates:
    assert len(resume_candidates) == 1, resume_candidates
    ckpt = torch.load(resume_candidates[0], map_location="cpu", weights_only=False)
    assert ckpt["experiment_id"] == EXPERIMENT_ID and ckpt["seed"] == SEED
    assert ckpt["protocol_sha256"] == sha256(PROTOCOL_PATH)
    assert ckpt["data_contract_sha256"] == sha256(DATA_CONTRACT_PATH)
    encoder.load_state_dict(ckpt["encoder_state_dict"], strict=True)
    decoder.load_state_dict(ckpt["decoder_state_dict"], strict=True)
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])
    random.setstate(ckpt["python_rng_state"])
    np.random.set_state(ckpt["numpy_rng_state"])
    torch.set_rng_state(ckpt["torch_rng_state"])
    torch.cuda.set_rng_state_all(ckpt["cuda_rng_state_all"])
    start_epoch, best_cider, stale, global_step = ckpt["epoch"] + 1, ckpt["best_cider"], ckpt["stale_evaluations"], ckpt["global_step"]
    resume_metrics_path = resume_candidates[0].parent / "metrics.csv"
    print("Exact resume           :", resume_candidates[0], "next epoch", start_epoch + 1)

history = []
if resume_metrics_path is not None and resume_metrics_path.exists():
    history = pd.read_csv(resume_metrics_path).to_dict("records")
elif METRICS_CSV.exists():
    history = pd.read_csv(METRICS_CSV).to_dict("records")

run_started = time.time()
for epoch in range(start_epoch, MAX_EPOCHS):
    epoch_started = time.time()
    print(
        f"Epoch {epoch+1:02d}/{MAX_EPOCHS} | TRAIN START | "
        f"batches={math.ceil(len(train_set) / 32)}",
        flush=True,
    )
    encoder.train()
    decoder.train()
    for module in encoder.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
    loader = make_train_loader(epoch)
    optimizer.zero_grad(set_to_none=True)
    loss_sum, count = 0.0, 0
    for micro_step, (_, images, tokens, mask) in enumerate(loader):
        images, tokens, mask = images.to(DEVICE), tokens.to(DEVICE), mask.to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16):
            memory = encoder(images)
            logits = decoder(tokens[:, :-1], memory, mask[:, :-1])
            targets = tokens[:, 1:]
            valid = mask[:, 1:]
            raw = F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]),
                                  targets.reshape(-1), reduction="none").reshape_as(targets)
            loss = (raw * valid).sum() / valid.sum()
        scaler.scale(loss / ACCUM).backward()
        loss_sum += float((raw.detach() * valid).sum())
        count += int(valid.sum())
        final_micro = micro_step + 1 == len(loader)
        if (micro_step + 1) % ACCUM == 0 or final_micro:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(list(encoder.parameters()) + list(decoder.parameters()), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1
        if (micro_step + 1) % 100 == 0 or final_micro:
            print(
                f"Epoch {epoch+1:02d}/{MAX_EPOCHS} | "
                f"batch={micro_step+1:04d}/{len(loader):04d} | "
                f"images={min((micro_step+1)*32, len(train_set)):05d}/{len(train_set):05d} | "
                f"elapsed={time.time()-epoch_started:.1f}s",
                flush=True,
            )
    print(
        f"Epoch {epoch+1:02d}/{MAX_EPOCHS} | VALIDATION START",
        flush=True,
    )
    metrics, hypotheses = validate()
    metrics.update({"epoch": epoch + 1, "train_ce": loss_sum / count,
                    "global_step": global_step, "elapsed_hours": (time.time() - run_started) / 3600})
    history.append(metrics)
    pd.DataFrame(history).to_csv(METRICS_CSV, index=False)
    improved = metrics["CIDEr_D"] > best_cider + MIN_DELTA
    if improved:
        best_cider, stale = metrics["CIDEr_D"], 0
    else:
        stale += 1
    save_checkpoint(LAST, epoch, best_cider, stale, global_step, metrics)
    if improved:
        save_checkpoint(BEST, epoch, best_cider, stale, global_step, metrics)
        (WORK_ROOT / "best_validation_captions.json").write_text(
            json.dumps({"references": VAL_REFERENCES, "hypotheses": hypotheses}, ensure_ascii=False),
            encoding="utf-8")
    print(f"Epoch {epoch+1:02d}/{MAX_EPOCHS} | train CE={metrics['train_ce']:.6f} | "
          f"val CE={metrics['ce_loss']:.6f} | CIDEr-D={metrics['CIDEr_D']:.6f} | "
          f"stale={stale}/{PATIENCE} | epoch_seconds={time.time()-epoch_started:.1f}",
          flush=True)
    if epoch + 1 >= MIN_EPOCHS and stale >= PATIENCE:
        print("Early stopping         : TRIGGERED")
        break

assert BEST.exists() and LAST.exists()
best_payload = torch.load(BEST, map_location="cpu", weights_only=False)
summary = {
    "schema_version": "f30k_student_baseline_result_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "complete",
    "experiment_id": EXPERIMENT_ID,
    "model_id": MODEL_ID,
    "model_name": MODEL_NAME,
    "seed": SEED,
    "best_epoch": int(best_payload["epoch"]) + 1,
    "best_validation_metrics": best_payload["metrics"],
    "selection_metric": "Validation CIDEr-D",
    "early_stopping_enabled": True,
    "maximum_epochs": MAX_EPOCHS,
    "test_set_status": "sealed",
    "test_set_used": False,
    "artifacts": {},
}
for path in [BEST, LAST, METRICS_CSV, WORK_ROOT / "best_validation_captions.json"]:
    summary["artifacts"][path.name] = {"bytes": path.stat().st_size, "sha256": sha256(path)}
(WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("Status                : COMPLETE")
print("Best epoch            :", summary["best_epoch"])
print("Best Validation       :", summary["best_validation_metrics"])
print("Output                :", WORK_ROOT)
print("Held-out Test         : SEALED / NOT USED")


Epoch 01/20 | TRAIN START | batches=907
Epoch 01/20 | batch=0100/0907 | images=03200/29000 | elapsed=395.0s
Epoch 01/20 | batch=0200/0907 | images=06400/29000 | elapsed=774.6s
Epoch 01/20 | batch=0300/0907 | images=09600/29000 | elapsed=1153.3s
Epoch 01/20 | batch=0400/0907 | images=12800/29000 | elapsed=1529.4s
Epoch 01/20 | batch=0500/0907 | images=16000/29000 | elapsed=1906.4s
Epoch 01/20 | batch=0600/0907 | images=19200/29000 | elapsed=2285.3s
Epoch 01/20 | batch=0700/0907 | images=22400/29000 | elapsed=2679.2s
Epoch 01/20 | batch=0800/0907 | images=25600/29000 | elapsed=3072.8s
Epoch 01/20 | batch=0900/0907 | images=28800/29000 | elapsed=3467.4s
Epoch 01/20 | batch=0907/0907 | images=29000/29000 | elapsed=3502.2s
Epoch 01/20 | VALIDATION START


PTBTokenizer tokenized 74030 tokens at 315877.37 tokens per second.
PTBTokenizer tokenized 15734 tokens at 133224.49 tokens per second.


{'testlen': 13824, 'reflen': 12137, 'guess': [13824, 12810, 11796, 10782], 'correct': [6504, 2402, 840, 315]}
ratio: 1.1389964571145144
Epoch 01/20 | train CE=5.517390 | val CE=4.133517 | CIDEr-D=0.169066 | stale=0/4 | epoch_seconds=3655.8
Epoch 02/20 | TRAIN START | batches=907
Epoch 02/20 | batch=0100/0907 | images=03200/29000 | elapsed=15.2s
Epoch 02/20 | batch=0200/0907 | images=06400/29000 | elapsed=30.0s
Epoch 02/20 | batch=0300/0907 | images=09600/29000 | elapsed=44.7s
Epoch 02/20 | batch=0400/0907 | images=12800/29000 | elapsed=59.7s
Epoch 02/20 | batch=0500/0907 | images=16000/29000 | elapsed=74.4s
Epoch 02/20 | batch=0600/0907 | images=19200/29000 | elapsed=89.2s
Epoch 02/20 | batch=0700/0907 | images=22400/29000 | elapsed=103.8s
Epoch 02/20 | batch=0800/0907 | images=25600/29000 | elapsed=118.8s
Epoch 02/20 | batch=0900/0907 | images=28800/29000 | elapsed=133.3s
Epoch 02/20 | batch=0907/0907 | images=29000/29000 | elapsed=134.2s
Epoch 02/20 | VALIDATION START


PTBTokenizer tokenized 74030 tokens at 343736.71 tokens per second.
PTBTokenizer tokenized 16007 tokens at 141064.56 tokens per second.


{'testlen': 13986, 'reflen': 12907, 'guess': [13986, 12972, 11958, 10944], 'correct': [7468, 3143, 1250, 514]}
ratio: 1.0835980475710014
Epoch 02/20 | train CE=3.706857 | val CE=3.699017 | CIDEr-D=0.249364 | stale=0/4 | epoch_seconds=220.1
Epoch 03/20 | TRAIN START | batches=907
Epoch 03/20 | batch=0100/0907 | images=03200/29000 | elapsed=15.2s
Epoch 03/20 | batch=0200/0907 | images=06400/29000 | elapsed=30.1s
Epoch 03/20 | batch=0300/0907 | images=09600/29000 | elapsed=44.9s
Epoch 03/20 | batch=0400/0907 | images=12800/29000 | elapsed=59.9s
Epoch 03/20 | batch=0500/0907 | images=16000/29000 | elapsed=74.4s
Epoch 03/20 | batch=0600/0907 | images=19200/29000 | elapsed=89.3s
Epoch 03/20 | batch=0700/0907 | images=22400/29000 | elapsed=104.1s
Epoch 03/20 | batch=0800/0907 | images=25600/29000 | elapsed=119.1s
Epoch 03/20 | batch=0900/0907 | images=28800/29000 | elapsed=134.0s
Epoch 03/20 | batch=0907/0907 | images=29000/29000 | elapsed=134.9s
Epoch 03/20 | VALIDATION START


PTBTokenizer tokenized 74030 tokens at 394086.06 tokens per second.
PTBTokenizer tokenized 14796 tokens at 133297.52 tokens per second.


... (31 outputs left out)

PTBTokenizer tokenized 74030 tokens at 327326.74 tokens per second.
PTBTokenizer tokenized 13681 tokens at 131357.80 tokens per second.


{'testlen': 11614, 'reflen': 11319, 'guess': [11614, 10600, 9586, 8572], 'correct': [7494, 3595, 1633, 773]}
ratio: 1.0260623730010578
Epoch 19/20 | train CE=2.526178 | val CE=2.918256 | CIDEr-D=0.476958 | stale=0/4 | epoch_seconds=210.5
Epoch 20/20 | TRAIN START | batches=907
Epoch 20/20 | batch=0100/0907 | images=03200/29000 | elapsed=15.1s
Epoch 20/20 | batch=0200/0907 | images=06400/29000 | elapsed=29.8s
Epoch 20/20 | batch=0300/0907 | images=09600/29000 | elapsed=44.6s
Epoch 20/20 | batch=0400/0907 | images=12800/29000 | elapsed=59.4s
Epoch 20/20 | batch=0500/0907 | images=16000/29000 | elapsed=74.1s
Epoch 20/20 | batch=0600/0907 | images=19200/29000 | elapsed=88.8s
Epoch 20/20 | batch=0700/0907 | images=22400/29000 | elapsed=103.5s
Epoch 20/20 | batch=0800/0907 | images=25600/29000 | elapsed=118.5s
Epoch 20/20 | batch=0900/0907 | images=28800/29000 | elapsed=133.2s
Epoch 20/20 | batch=0907/0907 | images=29000/29000 | elapsed=134.1s
Epoch 20/20 | VALIDATION START


PTBTokenizer tokenized 74030 tokens at 344212.20 tokens per second.
PTBTokenizer tokenized 13683 tokens at 114823.16 tokens per second.


{'testlen': 11609, 'reflen': 11300, 'guess': [11609, 10595, 9581, 8567], 'correct': [7480, 3568, 1607, 758]}
ratio: 1.0273451327432719
Epoch 20/20 | train CE=2.530288 | val CE=2.916928 | CIDEr-D=0.470112 | stale=1/4 | epoch_seconds=210.3
Status                : COMPLETE
Best epoch            : 19
Best Validation       : {'ce_loss': 2.918256190681752, 'BLEU_1': 0.6452557258480588, 'BLEU_4': 0.24079238427327226, 'METEOR': 0.20018132758052798, 'ROUGE_L': np.float64(0.46402429865452166), 'CIDEr_D': np.float64(0.4769577031590234), 'epoch': 19, 'train_ce': 2.5261777124603255, 'global_step': 8626, 'elapsed_hours': 2.0749728853835}
Output                : /kaggle/working/F30K_S1_2026
Held-out Test         : SEALED / NOT USED
